# Week 39, Day 3: Review: patterns

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w039d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Light:
    TABLE = {("off", "press"): "on", ("on", "press"): "dim", ("dim", "press"): "off", ("on", "hold"): "off", ("dim", "hold"): "off"}

    def __init__(self):
        self.state = "off"
        self.history = []

    def handle(self, event):
        new = self.TABLE.get((self.state, event))
        if new is None:
            raise ValueError(f"can't {event} when {self.state}")
        self.history.append(self.state)
        self.state = new
        return new

light = Light()
print([light.handle(e) for e in ["press", "press", "hold", "press"]], light.history)
try:
    light.handle("hold" if light.state == "off" else "x")
except ValueError as err:
    print(err)

In [ ]:
class Shop:
    def __init__(self, pricing):
        self.pricing = pricing                  # a strategy: any function (subtotal) -> total
        self.listeners = []                     # observers: functions (event, data)

    def subscribe(self, listener):
        self.listeners.append(listener)

    def checkout(self, items):
        total = self.pricing(sum(items))
        for listener in self.listeners:
            listener("sold", total)
        return total

def normal(subtotal):
    return subtotal

def sale(subtotal):
    return round(subtotal * 0.8)

log = []
shop = Shop(normal)
shop.subscribe(lambda event, total: log.append(f"{event}: {total}"))
shop.subscribe(lambda event, total: print("email: thank you for spending", total))
shop.checkout([500, 250])
shop.pricing = sale                              # swap the strategy at run time
shop.checkout([500, 250])
print(log)

In [ ]:
from functools import partial

class Exporter:
    registry = {}

    def __init_subclass__(cls, fmt, **kwargs):          # Week 35's self-registering factory
        super().__init_subclass__(**kwargs)
        Exporter.registry[fmt] = cls

    @classmethod
    def create(cls, fmt):
        try:
            return cls.registry[fmt]()
        except KeyError:
            raise ValueError(f"no exporter for {fmt!r}") from None

class CsvExporter(Exporter, fmt="csv"):
    def export(self, rows):
        return "\n".join(",".join(map(str, r)) for r in rows)

class TsvExporter(Exporter, fmt="tsv"):
    def export(self, rows):
        return "\n".join("\t".join(map(str, r)) for r in rows)

def send(channel, message):
    return f"[{channel}] {message}"

alert = partial(send, "alerts")                         # an adapter for callers that pass only a message

def counted(func):                                      # a decorator: same interface, extra behaviour
    def wrapper(*args):
        wrapper.calls += 1
        return func(*args)
    wrapper.calls = 0
    return wrapper

export = counted(Exporter.create("tsv").export)
print(repr(export([[1, 2], [3, 4]])), export.calls, sorted(Exporter.registry))
print(alert("disk full"))

In [ ]:
class Append:
    def __init__(self, doc, text):
        self.doc, self.text = doc, text

    def execute(self):
        self.doc.append(self.text)

    def undo(self):
        self.doc.pop()

class History:
    def __init__(self):
        self.done, self.undone = [], []

    def run(self, command):
        command.execute()
        self.done.append(command)
        self.undone.clear()                       # a new action makes the redo list meaningless

    def undo(self):
        if not self.done:
            return False
        command = self.done.pop()
        command.undo()
        self.undone.append(command)
        return True

    def redo(self):
        if not self.undone:
            return False
        command = self.undone.pop()
        command.execute()
        self.done.append(command)
        return True

doc, h = [], History()
for word in ["one", "two", "three"]:
    h.run(Append(doc, word))
h.undo(); h.undo(); h.redo()
print(doc, len(h.done), len(h.undone))
h.run(Append(doc, "four"))
print(doc, h.redo())

**Predict before you run:** what will this print?



In [ ]:
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return CountdownIterator(self.start)

class CountdownIterator:
    def __init__(self, n):
        self.n = n

    def __iter__(self):
        return self

    def __next__(self):
        if self.n <= 0:
            raise StopIteration
        self.n -= 1
        return self.n + 1

c = Countdown(3)
it = iter(c)
print(next(it), list(it), list(c), list(it))

In [ ]:
from datetime import date

class MemoryNotes:
    def __init__(self):
        self.notes = {}

    def add(self, key, text):
        self.notes[key] = text

    def all(self):
        return dict(self.notes)

class NoteService:
    def __init__(self, repo, today):
        self.repo, self.today = repo, today

    def write(self, text):
        if not text.strip():
            raise ValueError("empty note")
        key = f"{self.today().isoformat()}-{len(self.repo.all()) + 1}"
        self.repo.add(key, text.strip())
        return key

def build(settings):                            # the composition root
    return NoteService(MemoryNotes(), settings.get("clock", date.today))

service = build({"clock": lambda: date(2026, 11, 2)})   # a fixed clock, as a test would use
print(service.write(" buy tea "), service.write("call Ada"), service.repo.all())

> **Key idea.** Strategy swaps one step (usually a function); Observer tells subscribers about events; Factory chooses the class (a dictionary or `__init_subclass__` registry); Adapter translates an interface (`partial` for arguments); Decorator wraps with the same interface. State moves through a transition table; Command objects `execute` and `undo`, with two stacks; Iterator classes give a fresh iterator per loop. Programs are layered, with dependencies pointing inwards to the domain, repositories for storage, services for use cases, and everything injected from a composition root.

### Exercise: Sorting strategies

Write `rank(players, by)`, where `players` is a list of `(name, points, wins)` tuples and `by` names a strategy in a dictionary `STRATEGIES`: `"points"` (most points first), `"wins"` (most wins first, then most points), and `"name"` (alphabetical, ignoring case). Ties keep their original order. An unknown strategy raises `ValueError("no strategy 'X'")`. Each strategy is a key function in the dictionary.

In [ ]:
STRATEGIES = {
    ___
}

def rank(players, by):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("P = [(\"cara\", 10, 3), (\"Ade\", 12, 2), (\"bo\", 10, 4), (\"Dee\", 12, 3)]\ndef test_rank():\n    \"strategies\"\n    assert [p[0] for p in rank(P, \"points\")] == [\"Ade\", \"Dee\", \"cara\", \"bo\"]\n    assert [p[0] for p in rank(P, \"wins\")] == [\"bo\", \"Dee\", \"cara\", \"Ade\"]\n    assert [p[0] for p in rank(P, \"name\")] == [\"Ade\", \"bo\", \"cara\", \"Dee\"]\n    assert all(callable(f) for f in STRATEGIES.values()) and len(STRATEGIES) == 3\n    try:\n        rank(P, \"luck\")\n        assert False\n    except ValueError as err:\n        assert str(err) == \"no strategy 'luck'\"")

### Exercise: A thermostat with observers

Write `Thermostat(target)` with `subscribe(listener)` and `unsubscribe(listener)`, and `reading(temp)`, which records the temperature and, if it is more than 2 degrees away from the target, calls each listener with `("too hot", temp)` or `("too cold", temp)`. A listener may unsubscribe **itself** while being called, which must not make the thermostat skip anyone else. `reading` returns how many listeners were told.

In [ ]:
class Thermostat:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_alerts():\n    \"alerts\"\n    t = Thermostat(20)\n    got = []\n    t.subscribe(lambda kind, temp: got.append((kind, temp)))\n    assert t.reading(21) == 0 and t.reading(23) == 1 and t.reading(17.5) == 1 and t.reading(22) == 0\n    assert got == [(\"too hot\", 23), (\"too cold\", 17.5)]\ndef test_unsubscribe_while_called():\n    \"unsubscribing during a call\"\n    t = Thermostat(20)\n    seen = []\n    def once(kind, temp):\n        seen.append(\"once\")\n        t.unsubscribe(once)\n    t.subscribe(once)\n    t.subscribe(lambda k, x: seen.append(\"always\"))\n    assert t.reading(30) == 2 and seen == [\"once\", \"always\"]\n    assert t.reading(30) == 1 and seen == [\"once\", \"always\", \"always\"]")

### Exercise: A registering factory

Write a base class `Shape` whose subclasses register themselves by name with `__init_subclass__(cls, name, **kwargs)`, and a class method `Shape.create(spec)` that reads text like `"circle 2"` or `"rect 3 4"`: the name, then numbers (as floats) passed to the class. An unknown name raises `ValueError("unknown shape 'NAME'")`. Write `Circle` (name `"circle"`, `area()` with `math.pi`) and `Rect` (name `"rect"`) as subclasses.

In [ ]:
import math

class Shape:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_factory():\n    \"the factory\"\n    assert round(Shape.create(\"circle 2\").area(), 3) == 12.566 and Shape.create(\"rect 3 4\").area() == 12.0\n    assert isinstance(Shape.create(\"rect 1 1\"), Rect)\n    class Tri(Shape, name=\"tri\"):\n        def __init__(self, b, h):\n            self.b, self.h = b, h\n        def area(self):\n            return self.b * self.h / 2\n    assert Shape.create(\"tri 4 3\").area() == 6.0, \"a new subclass registers itself\"\n    try:\n        Shape.create(\"hexagon 1\")\n        assert False\n    except ValueError as err:\n        assert str(err) == \"unknown shape 'hexagon'\"")

### Exercise: Undoable edits

A "document" is a list of lines. Write two commands, `Insert(doc, index, line)` and `Delete(doc, index)`, each with `execute()` and `undo()` that put the document back **exactly** (Delete must remember what it removed). Then `Editor(doc)` with `do(command)`, `undo()` and `redo()` (both returning `False` when there's nothing to do), with the redo stack cleared by any new `do`.

In [ ]:
class Insert:
    ___

class Delete:
    ___

class Editor:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_editor():\n    \"undo and redo\"\n    doc = [\"a\", \"b\", \"c\"]\n    e = Editor(doc)\n    e.do(Insert(doc, 1, \"x\"))\n    e.do(Delete(doc, 3))\n    assert doc == [\"a\", \"x\", \"b\"]\n    assert e.undo() and doc == [\"a\", \"x\", \"b\", \"c\"]\n    assert e.undo() and doc == [\"a\", \"b\", \"c\"] and not e.undo()\n    assert e.redo() and doc == [\"a\", \"x\", \"b\", \"c\"]\n    e.do(Delete(doc, 0))\n    assert doc == [\"x\", \"b\", \"c\"] and not e.redo(), \"a new command clears redo\"\n    assert e.undo() and doc == [\"a\", \"x\", \"b\", \"c\"]")

### Bug hunt: Only once round

`Week` should be loopable as often as you like, but its `__iter__` returns `self`, so the second loop finds it already used up and prints `[]`. Give it a separate iterator, so the output is `['Mon', 'Tue', 'Wed']` twice.

In [ ]:
class Week:
    def __init__(self, days):
        self.days = days
        self.i = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.i >= len(self.days):
            raise StopIteration
        self.i += 1
        return self.days[self.i - 1]

w = Week(["Mon", "Tue", "Wed"])
print(list(w))
print(list(w))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a fresh iterator each time\"\n    expect_output(\"['Mon', 'Tue', 'Wed']\\n['Mon', 'Tue', 'Wed']\")")

### Bug hunt: Redo after moving on

After undoing, the user types something new, then presses redo. The old command is replayed onto a document that has moved on, giving `['a', 'c', 'b']`. A new action must clear the redo stack, so the output is `['a', 'c']` then `False`.

In [ ]:
doc = []
done, undone = [], []

def do(text):
    doc.append(text)
    done.append(text)

def undo():
    undone.append(done.pop())
    doc.pop()

def redo():
    if not undone:
        return False
    text = undone.pop()
    doc.append(text)
    done.append(text)
    return True

do("a"); do("b")
undo()
do("c")
result = redo()
print(doc)
print(result)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"clear redo\"\n    expect_output(\"['a', 'c']\\nFalse\")")

### Bug hunt: A hidden clock

`Reminder` uses the real clock inside `is_due`, so it can't be checked on a chosen day: this should show that the reminder isn't due on 1 November, but it uses today's date instead. Inject the clock (a parameter `today`, a function returning a date, defaulting to `date.today`, stored as `self.today`), and pass a fixed one, so the program prints `False`.

In [ ]:
from datetime import date

class Reminder:
    def __init__(self, due):
        self.due = due

    def is_due(self):
        return date.today() >= self.due

r = Reminder(date(2026, 11, 2))         # the day before: 1 November
print(r.is_due())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an injected clock\"\n    expect_output(\"False\")\n    from datetime import date as d\n    assert Reminder(d(2026, 11, 2), today=lambda: d(2026, 11, 1)).is_due() is False\n    assert Reminder(d(2026, 11, 2)).today == d.today")

### Puzzle: An adapter

Arrange the lines into an adapter that lets an old printer, which has only `print_text(text, copies)`, be used where code calls `send(message)`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from functools import partial

def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube = partial(power, exp=3)

def twice(f):
    def wrapper(x):
        return f(f(x))
    return wrapper

quad = twice(square)

class Nums:
    def __init__(self, n):
        self.n = n
    def __iter__(self):
        return iter(range(self.n))

p = ___   # square(5)
q = ___   # cube(2)
r = ___   # quad(3)
s = ___   # sum(Nums(4)) + sum(Nums(4))
t = ___   # list(zip(Nums(2), Nums(3)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 25 and q == 8 and r == 81, \"p, q and r\"\n    assert s == 12 and t == [(0, 0), (1, 1)], \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a vending machine

Build a vending machine with the State pattern, observers and an injected stock repository:

- `MemoryStock(items)` takes a dictionary from code to `(name, price_pence, count)` and has `get(code)` (the tuple, or `KeyError`) and `take(code)` (reduces the count by 1).
- `Machine(stock)` starts in state `"idle"` with `credit` 0. Methods:
  - `insert(pence)`: allowed in `"idle"` and `"paying"`; adds to the credit and moves to `"paying"`.
  - `choose(code)`: allowed only in `"paying"`. An unknown code, or one with count 0, raises `ValueError` and changes nothing. Not enough credit raises `ValueError("insert N more")` (N in pence). Otherwise it takes the item, keeps the change in credit, notifies observers with `("sold", name)`, and moves to `"dispensing"`.
  - `collect()`: allowed only in `"dispensing"`; returns `(name, change)`, resets credit to 0, and moves to `"idle"`.
  - `cancel()`: allowed in `"paying"`; returns the credit, sets it to 0, and moves to `"idle"`.
  - Any method called in a state that doesn't allow it raises `RuntimeError("can't X while STATE")`, where X is the method's name.
  - `subscribe(listener)` adds an observer.

In [ ]:
class MemoryStock:
    ___

class Machine:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make():\n    return Machine(MemoryStock({\"A1\": (\"Crisps\", 80, 2), \"B2\": (\"Water\", 120, 0)}))\ndef test_sale():\n    \"a sale\"\n    m, events = make(), []\n    m.subscribe(lambda kind, name: events.append((kind, name)))\n    assert m.state == \"idle\"\n    m.insert(50)\n    m.insert(50)\n    assert m.state == \"paying\" and m.credit == 100\n    m.choose(\"A1\")\n    assert m.state == \"dispensing\" and events == [(\"sold\", \"Crisps\")] and m.stock.get(\"A1\")[2] == 1\n    assert m.collect() == (\"Crisps\", 20) and m.state == \"idle\" and m.credit == 0\ndef test_refusals():\n    \"refusals\"\n    m = make()\n    m.insert(50)\n    for code, words in [(\"A1\", \"insert 30 more\"), (\"B2\", \"\"), (\"Z9\", \"\")]:\n        try:\n            m.choose(code)\n            assert False, code\n        except ValueError as err:\n            assert words in str(err)\n    assert m.state == \"paying\" and m.credit == 50 and m.stock.get(\"A1\")[2] == 2\n    assert m.cancel() == 50 and m.state == \"idle\"\ndef test_states():\n    \"states\"\n    m = make()\n    for call, message in [(lambda: m.choose(\"A1\"), \"can't choose while idle\"), (m.collect, \"can't collect while idle\"), (m.cancel, \"can't cancel while idle\")]:\n        try:\n            call()\n            assert False, message\n        except RuntimeError as err:\n            assert str(err) == message\n    m.insert(100)\n    m.choose(\"A1\")\n    for call, message in [(lambda: m.insert(10), \"can't insert while dispensing\"), (m.cancel, \"can't cancel while dispensing\")]:\n        try:\n            call()\n            assert False\n        except RuntimeError as err:\n            assert str(err) == message")

## Warm-up

A state machine as a transition table (Week 36): every allowed move is one entry, and anything else is refused with a clear error.

## Behaviour you can swap

| Pattern | The problem | The Python shape |
|---|---|---|
| **Strategy** | one step of an algorithm varies (how to price, sort, compress) | pass in a function or an object with one method; often just a dictionary of functions |
| **Observer** | when something happens, several other parts need to know, and the source shouldn't know who they are | the subject keeps a list of callbacks and calls each on an event; subscribers can be added and removed |

In Python a strategy is usually a **function**: there's no need for a class with one method unless the strategy carries settings or state.

## Making and wrapping

| Pattern | The problem | The Python shape |
|---|---|---|
| **Factory** | callers shouldn't need to know which class to make | a function or dictionary from a name to a class; `__init_subclass__` lets subclasses register themselves |
| **Adapter** | an object has the right abilities but the wrong interface | a small class that holds it and translates calls; `functools.partial` for argument mismatches |
| **Decorator** (the pattern) | add behaviour (logging, caching, checking) to an object or function without changing it | a wrapper with the same interface that calls the original: `f = deco(f)` |

The `@` syntax for decorators (`@counted` above a `def`) is shorthand for `f = counted(f)`; Phase 5 covers it properly, with closures.

## State, Command and Iterator

| Pattern | The problem | The Python shape |
|---|---|---|
| **State** | behaviour depends on a status, and `if status ==` spreads through every method | a transition table, or one class per state, each answering for itself |
| **Command** | actions must be stored, undone, redone, queued or logged | objects with `execute()` and `undo()`; two stacks for undo and redo; a macro holds a list of commands |
| **Iterator** | walk through a structure without exposing how it's stored | `__iter__` returning an object with `__next__` that raises `StopIteration`; an iterable gives a **fresh** iterator each time |

The iterable (`Countdown`) can be looped over again and again; each **iterator** is used up once.

## Whole programs

Week 36, and `shelf` in Week 38:

- **Layers**: presentation (input and output) → application (use cases, in a service) → domain (the rules) ← infrastructure (storage, the outside world). Dependencies point **inwards**: the domain imports nothing from the others.
- **Repository**: the one place that knows how objects are stored, with methods like `get`, `add`, `save`; a memory version for tests, a file or database version for real.
- **Service**: one method per use case; reads input, finds objects, asks the domain, commits; never prints.
- **Dependency injection**: objects receive what they need (store, clock, sender) instead of creating it; a **composition root** builds the real program from settings, and tests build it with fakes.

## The classic mistakes

1. **A pattern where a function would do**: a `PricingStrategy` class hierarchy for three one-line rules. KISS.
2. **Observers changing the subject's list while it loops over it**: loop over a copy (`list(self.listeners)`).
3. **Not clearing the redo stack** after a new command: redo then applies a command to a document that has moved on.
4. **An iterable that returns `self` from `__iter__`**: it can only be looped over once.
5. **The domain importing from storage or the interface**: the dependency rule broken, and the rules can't be tested alone.
6. **Creating dependencies inside a class** (`self.clock = date.today`): hidden, and impossible to replace in tests.

## Practice

## Quiz

## Daily challenge

## Before you go

For each of the eight patterns, write its name, the problem it solves in one sentence, and where you used it in `shelf` or the Forges. Tomorrow: a practice exam with the same shape as Friday's.